# Dataset audit and model comparison
Research question: does machine learning add information beyond AQ scores when reproducing this dataset's screening labels?

This adult dataset does not establish clinical outcomes. All comparisons below are exploratory. A score threshold of 7 is a **dataset-label reconstruction baseline**, not clinical guidance. Checking agreement does not establish how labels were collected.

Run all cells in order. Outputs go to `research/results/`; deployed model artifacts are not overwritten.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/train.py").exists())
DATA = ROOT / "data/train.csv"
OUT = ROOT / "research/archive/preliminary/results"
AQ = [f"A{i}_Score" for i in range(1, 11)]
CAT = ["gender", "ethnicity", "jaundice", "country_of_res", "relation"]
NUM = AQ + ["age", "sum_score"]
raw = pd.read_csv(DATA, keep_default_na=False)
raw = raw.rename(columns={"jundice": "jaundice", "contry_of_res": "country_of_res"})
print("Dataset:", DATA, "| shape:", raw.shape)


## 1. Raw data quality
Count empty strings and question marks before replacing them. Flag ages outside 18–100 as an audit convention; do not silently remove or correct them. Keep questionable rows in the primary experiment and report their presence.


In [ ]:
missing = raw.apply(lambda col: col.astype(str).str.strip().isin(["", "?"]))
missing_counts = missing.sum().sort_values(ascending=False)
aq_numeric = raw[AQ].apply(pd.to_numeric, errors="coerce")
invalid_aq = ~aq_numeric.isin([0, 1])
age = pd.to_numeric(raw["age"], errors="coerce")
age_flags = age.isna() | ~age.between(18, 100)
labels = raw["Class/ASD"].astype(str).str.strip().str.upper()
assert labels.isin(["NO", "YES"]).all(), "Unexpected target labels"
assert not invalid_aq.any().any(), "Resolve missing/invalid AQ values before modeling"
score = aq_numeric.sum(axis=1)
y = labels.map({"NO": 0, "YES": 1})
rule = (score >= 7).astype(int)
audit = {
    "rows": len(raw),
    "class_counts": labels.value_counts().to_dict(),
    "missing_by_column": {k: int(v) for k, v in missing_counts.items()},
    "invalid_aq_cells": int(invalid_aq.sum().sum()),
    "duplicate_full_rows": int(raw.duplicated().sum()),
    "age_flagged_rows": int(age_flags.sum()),
    "age_min": float(age.min()), "age_max": float(age.max()),
    "score_rule_disagreements": int((rule != y).sum()),
    "result_differs_from_sum": int((pd.to_numeric(raw["result"], errors="coerce") != score).sum()),
}
print(json.dumps(audit, indent=2))
display(raw.loc[age_flags, ["age", "Class/ASD"]])
display(pd.crosstab(score.rename("AQ sum"), labels))
display(raw.loc[rule != y, AQ + ["Class/ASD"]])


## 2. Prepare predictors and shared validation folds
Exclude the target, recorded result, IDs, and other unused fields. Imputation, encoding, and scaling are fitted only within each training fold. Missing and flagged ages become missing numeric values for median imputation; raw records remain available above.

Group identical combined predictor records so they cannot cross folds. These groups represent matching records, not verified participant identities. Without participant IDs, repeated people with different records could still cross folds.


In [ ]:
X = raw[AQ + ["age"] + CAT].copy()
X[AQ] = aq_numeric.astype(int)
X["age"] = age.mask(age_flags)
for col in CAT:
    X[col] = X[col].astype(str).str.strip().replace({"?": np.nan, "": np.nan, "others": "Others"})
X["sum_score"] = score
keys = X.fillna("<missing>").astype(str).agg("|".join, axis=1)
groups = pd.factorize(keys)[0]
audit["duplicate_predictor_rows"] = int(keys.duplicated().sum())
audit["conflicting_predictor_groups"] = int(pd.DataFrame({"key": keys, "label": y}).groupby("key")["label"].nunique().gt(1).sum())
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(cv.split(X, y, groups))
for train, test in splits:
    assert set(groups[train]).isdisjoint(groups[test])
    assert y.iloc[train].nunique() == y.iloc[test].nunique() == 2
print("Unique predictor groups:", len(set(groups)))
print("Validation sizes:", [len(test) for _, test in splits])


## 3. Model comparison
Use identical folds for all methods. Compare majority class, the fixed score rule, and logistic regression / decision tree models for three feature sets. Questionnaire-only models exclude the summed score to test what the individual items provide. Combined models include the sum, matching the app's feature choice.

No hyperparameter search is performed. Out-of-fold predictions support exploratory comparison; choosing a winner here requires a fresh evaluation dataset. Probability metrics describe agreement with screening labels, not the probability of an autism diagnosis.


In [ ]:
feature_sets = {
    "questionnaire": (AQ, []),
    "demographics": (["age"], CAT),
    "combined": (NUM, CAT),
}
def make_pipeline(kind, numeric, categorical):
    transforms = []
    if numeric:
        transforms.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric))
    if categorical:
        transforms.append(("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical))
    classifier = LogisticRegression(max_iter=2000, random_state=42) if kind == "logistic" else DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
    return Pipeline([("preprocess", ColumnTransformer(transforms)), ("classifier", classifier)])

def metrics(target, pred, prob):
    tn, fp, fn, tp = confusion_matrix(target, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(target, pred),
            "precision": precision_score(target, pred, zero_division=0),
            "recall": recall_score(target, pred, zero_division=0),
            "specificity": tn / (tn + fp),
            "f1": f1_score(target, pred, zero_division=0),
            "roc_auc": roc_auc_score(target, prob),
            "brier": brier_score_loss(target, prob),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

names = ["majority", "score_rule"] + [f"{kind}_{features}" for features in feature_sets for kind in ["logistic", "tree"]]
rows, fold_rows, predictions = [], [], {}
for name in names:
    oof_prob, oof_pred = np.zeros(len(X)), np.zeros(len(X), dtype=int)
    for fold, (train, test) in enumerate(splits, 1):
        if name == "score_rule":
            pred = rule.iloc[test].to_numpy()
            prob = pred.astype(float)
        elif name == "majority":
            estimator = DummyClassifier(strategy="most_frequent").fit(X.iloc[train], y.iloc[train])
            pred = estimator.predict(X.iloc[test])
            prob = estimator.predict_proba(X.iloc[test])[:, 1]
        else:
            kind, features = name.split("_", 1)
            estimator = make_pipeline(kind, *feature_sets[features])
            estimator.fit(X.iloc[train], y.iloc[train])
            pred = estimator.predict(X.iloc[test])
            prob = estimator.predict_proba(X.iloc[test])[:, 1]
        oof_pred[test], oof_prob[test] = pred, prob
        fold_rows.append({"method": name, "fold": fold, **metrics(y.iloc[test], pred, prob)})
    predictions[name] = (oof_pred, oof_prob)
    rows.append({"method": name, **metrics(y, oof_pred, oof_prob)})
comparison = pd.DataFrame(rows).set_index("method")
fold_metrics = pd.DataFrame(fold_rows)
display(comparison.round(4))
display(fold_metrics.groupby("method")[["accuracy", "recall", "specificity", "f1", "brier"]].agg(["mean", "std"]).round(4))


## 4. Calibration and error inspection
Inspect calibration bins and confusion counts without adding plotting dependencies. Small bins have substantial uncertainty. A deterministic score rule has only two probability values and is not a calibrated clinical probability model.


In [ ]:
calibration_rows = []
for name, (pred, prob) in predictions.items():
    observed, predicted = calibration_curve(y, prob, n_bins=5, strategy="uniform")
    calibration_rows.extend({"method": name, "mean_predicted": float(p), "observed_positive_fraction": float(o)} for p, o in zip(predicted, observed))
calibration = pd.DataFrame(calibration_rows)
display(calibration)
pred, prob = predictions["logistic_combined"]
errors = raw.loc[pred != y, AQ + ["age", "Class/ASD"]].copy()
errors["predicted_label"] = pred[pred != y]
errors["model_probability"] = prob[pred != y]
display(errors)


## 5. Export reproducibility artifacts
Record dataset checksum, software versions, fold assignment, and all evaluation tables. These generated files contain public dataset row indices and model outputs; do not substitute private participant records without revisiting the export policy.


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUT / "model_comparison.csv")
fold_metrics.to_csv(OUT / "fold_metrics.csv", index=False)
calibration.to_csv(OUT / "calibration.csv", index=False)
fold_assignment = np.zeros(len(X), dtype=int)
for fold, (_, test) in enumerate(splits, 1):
    fold_assignment[test] = fold
pd.DataFrame({"row_index": np.arange(len(X)), "group": groups, "fold": fold_assignment}).to_csv(OUT / "fold_assignment.csv", index=False)
provenance = {"sha256": hashlib.sha256(DATA.read_bytes()).hexdigest(),
              "python": platform.python_version(), "pandas": pd.__version__,
              "numpy": np.__version__, "sklearn": sklearn.__version__,
              "seed": 42, "validation": "5-fold StratifiedGroupKFold on identical combined predictors",
              "age_cleaning": "Outside 18-100 treated as missing; median imputed within training folds",
              "score_rule": "sum_score >= 7; dataset reconstruction baseline"}
(OUT / "audit.json").write_text(json.dumps(audit, indent=2))
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=2))
print("Results saved to", OUT)


## Interpretation for your paper
- If the score rule matches every label, the experiment demonstrates label reconstruction. It does not establish independent clinical prediction or prove the original label-generation procedure.
- Compare demographics-only performance against the majority baseline, and describe potential sampling effects.
- Compare questionnaire models against the score rule before claiming added value from AI.
- Fold standard deviations describe variation across folds; they are not confidence intervals.
- This analysis does not include external validation, diagnosed clinical outcomes, subgroup fairness conclusions, or pediatric generalization.
- The primary experiment retains records with questionable ages after marking those ages missing. A future sensitivity analysis should repeat the comparison after excluding those records.
- Write conclusions from the generated tables, including negative findings. Do not promote experiment models into the app automatically.
